# Jackett Heroku Deployer

This automated Google Colab notebook will guide you through deploying Jackett to Heroku natively using Heroku's Git-based deployment.

**Note:** This notebook uses Heroku's container stack combined with Git push deployment (`heroku.yml`). This avoids needing Docker daemon installed in Colab, while still allowing Heroku to build and host the Dockerfile safely!

**Instructions:**
Run each cell sequentially (Shift+Enter). Provide the requested information when prompted.

## Step 1: Install Dependencies
This step installs the necessary Linux tools and the Heroku CLI. Colab does not persist these between sessions, so this must be run every time.
* Expected duration: ~30 seconds

In [ ]:
import subprocess

print('⏳ Installing dependencies... (This takes about 30 seconds)')
subprocess.run(['apt-get', 'update'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
subprocess.run(['apt-get', 'install', '-y', 'jq', 'curl', 'git'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Install Heroku CLI securely
install_script = subprocess.run(['curl', '-sL', 'https://cli-assets.heroku.com/install-ubuntu.sh'], capture_output=True, text=True)
if install_script.returncode == 0:
    subprocess.run(['bash', '-c', install_script.stdout], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    print('⚠️ Warning: Failed to download Heroku CLI installer.')

subprocess.run(['pip', 'install', '-q', 'ipywidgets', 'requests'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print('✅ Dependencies installed successfully.')

## Step 2: Collect User Input
Please provide your configuration variables. Your secrets are hidden and only used within this session securely.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

print("Fill out the forms below, then proceed to the next cell.")

api_key_input = widgets.Password(description='Heroku API Key:', style={'description_width': 'initial'})
team_name_input = widgets.Text(description='Team Name (Optional):', placeholder='Leave blank for personal apps', style={'description_width': 'initial'})
app_name_input = widgets.Text(description='App Name:', placeholder='my-jackett-app', style={'description_width': 'initial'})
region_input = widgets.Dropdown(options=['us', 'eu'], value='us', description='Region:', style={'description_width': 'initial'})

repo_url_input = widgets.Text(description='GitHub Repo URL:', value='https://github.com/Jackett/Jackett.git', style={'description_width': 'initial'})
branch_input = widgets.Text(description='Branch:', value='master', style={'description_width': 'initial'})
git_token_input = widgets.Password(description='GitHub PAT (Optional):', placeholder='For private repos', style={'description_width': 'initial'})

verbose_input = widgets.Checkbox(value=False, description='Verbose Logging')

display(widgets.VBox([
    widgets.HTML("<b>Heroku Configuration</b>"),
    api_key_input, team_name_input, app_name_input, region_input,
    widgets.HTML("<br><b>GitHub Configuration</b>"),
    repo_url_input, branch_input, git_token_input,
    widgets.HTML("<br><b>Options</b>"),
    verbose_input
]))

## Step 3: Authenticate with Heroku
Using your provided API key, we will log into Heroku.
* Expected duration: ~5 seconds

In [ ]:
import os
import subprocess

api_key = api_key_input.value.strip()
if not api_key:
    raise ValueError('❌ Heroku API Key is required.')

# Create .netrc for Heroku authentication
with open(os.path.expanduser('~/.netrc'), 'w') as f:
    f.write(f"machine api.heroku.com\n  login {api_key}\n  password {api_key}\n")
    f.write(f"machine git.heroku.com\n  login {api_key}\n  password {api_key}\n")

os.environ['HEROKU_API_KEY'] = api_key

result = subprocess.run(['heroku', 'auth:whoami'], capture_output=True, text=True)

if result.returncode != 0:
    print("❌ Authentication failed. Please check your API key.")
    print(result.stderr)
    raise Exception('Authentication failed.')
else:
    print(f"✅ Successfully authenticated as: {result.stdout.strip()}")
    
# Fetch Team Memberships
teams_result = subprocess.run(['heroku', 'teams'], capture_output=True, text=True)
if teams_result.returncode == 0:
    print("\nYour Team Memberships:")
    print(teams_result.stdout.strip())

## Step 4: Clone the GitHub Repository
We'll clone your specified GitHub repository into the Colab environment.
* Expected duration: ~10-20 seconds

In [ ]:
import shutil

repo_url = repo_url_input.value.strip()
branch = branch_input.value.strip()
git_token = git_token_input.value.strip()

if not repo_url or not branch:
    raise ValueError("❌ Repository URL and Branch are required.")

if git_token:
    if repo_url.startswith("https://"):
        repo_url = repo_url.replace("https://", f"https://{git_token}@")

if os.path.exists('jackett_src'):
    shutil.rmtree('jackett_src')

print(f"⏳ Cloning {repo_url_input.value} (Branch: {branch})...")
clone_result = subprocess.run(['git', 'clone', '-b', branch, repo_url, 'jackett_src'], capture_output=True, text=True)

if clone_result.returncode != 0:
    print("❌ Failed to clone repository. Check your URL, branch, or PAT.")
    if verbose_input.value: print(clone_result.stderr)
    raise Exception('Clone failed')

os.chdir('jackett_src')

commit_result = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True)
print("✅ Repository cloned successfully.")
print(f"Current Commit: {commit_result.stdout.strip()}")

## Step 5: Create or Connect to Heroku App
We will create the app on Heroku. If it already exists, we will update the deployment config.
* Expected duration: ~5 seconds

In [ ]:
app_name = app_name_input.value.strip()
team_name = team_name_input.value.strip()
region = region_input.value

if not app_name:
    raise ValueError("❌ App Name is required.")

print(f"⏳ Checking if app '{app_name}' exists...")
check_result = subprocess.run(['heroku', 'info', '-a', app_name], capture_output=True, text=True)

if check_result.returncode == 0:
    print(f"✅ App '{app_name}' already exists. We will reuse it.")
else:
    print(f"⏳ Creating app '{app_name}' in region '{region}'...")
    cmd = ['heroku', 'apps:create', app_name, '--region', region]
    if team_name:
        cmd.extend(['--team', team_name])
    
    create_result = subprocess.run(cmd, capture_output=True, text=True)
    if create_result.returncode != 0:
        print("❌ Failed to create app.")
        print(create_result.stderr)
        raise Exception('App creation failed')
    print("✅ App created successfully.")

# Set stack to container
print("⏳ Setting Heroku stack to 'container'...")
stack_result = subprocess.run(['heroku', 'stack:set', 'container', '-a', app_name], capture_output=True, text=True)
if stack_result.returncode != 0:
    print("❌ Failed to set stack.")
    print(stack_result.stderr)
    raise Exception('Stack configuration failed')
print("✅ Stack configured to 'container'.")

## Step 6: Configure Environment Variables
Configuring `HEROKU=true` and `ASPNETCORE_ENVIRONMENT=Production`.

In [ ]:
print("⏳ Applying config vars...")
config_result = subprocess.run(['heroku', 'config:set', 'HEROKU=true', 'ASPNETCORE_ENVIRONMENT=Production', '-a', app_name], capture_output=True, text=True)

if config_result.returncode != 0:
    print("❌ Failed to set config vars.")
    print(config_result.stderr)
    raise Exception('Config vars failed')
print("✅ Config vars applied successfully.")

## Step 7: Configure Git Remote
Binding Heroku Git remote for deployment.

In [ ]:
print("⏳ Configuring Heroku git remote...")
remote_result = subprocess.run(['heroku', 'git:remote', '-a', app_name], capture_output=True, text=True)
if remote_result.returncode != 0:
    print("❌ Failed to add git remote.")
    print(remote_result.stderr)
    raise Exception('Git remote failed')

print("✅ Remote configured:")
result = subprocess.run(['git', 'remote', '-v'], capture_output=True, text=True)
print(result.stdout)

## Step 8: Deploy
We'll push the repository to Heroku. Heroku will build the Dockerfile using Heroku Git.
* Expected duration: ~3-5 minutes

In [ ]:
import sys

print("⏳ Pushing code to Heroku. This triggers a build and takes a few minutes. Please wait...")

# We use Popen to stream the output directly to the notebook cell
process = subprocess.Popen(['git', 'push', 'heroku', f'HEAD:main', '-f'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

for line in process.stdout:
    sys.stdout.write(line)

process.wait()

if process.returncode != 0:
    print("\n❌ Deployment failed.")
    raise Exception('Git push failed')
else:
    print("\n✅ Build and push completed successfully.")

## Step 9: Verify Deployment
Wait for the dyno to boot and verify the `/health` endpoint.
* Expected duration: ~15-30 seconds

In [ ]:
import time
import requests
from datetime import datetime

url = f"https://{app_name}.herokuapp.com"
health_url = f"{url}/health"

print("⏳ Waiting for the dyno to boot (20 seconds)...")
time.sleep(20)

try:
    response = requests.get(health_url, timeout=15)
    if response.status_code == 200 and 'OK' in response.text:
        print("\n🎉 **Deployment Summary**")
        print(f"✅ Status: Successful")
        print(f"🔗 App URL: {url}")
        print(f"🏥 Health URL: {health_url}")
        print(f"📊 Dashboard URL: https://dashboard.heroku.com/apps/{app_name}")
        print(f"⏱️ Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"🏷️ Branch: {branch}")
    else:
        print("\n⚠️ Deployment might have failed or app is still booting.")
        print(f"Status Code: {response.status_code}, Response: {response.text}")
        print("\n--- Recent Logs ---")
        subprocess.run(['heroku', 'logs', '--tail', '-n', '30', '-a', app_name])
except Exception as e:
    print("\n❌ Failed to connect to health endpoint.")
    print(str(e))
    print("\n--- Recent Logs ---")
    subprocess.run(['heroku', 'logs', '--tail', '-n', '30', '-a', app_name])

## Step 10: Diagnostics (Run if Step 9 failed)
If the deployment failed, run this cell to see detailed diagnostics.

In [ ]:
print("--- Current Config Vars ---")
config = subprocess.run(['heroku', 'config', '-a', app_name], capture_output=True, text=True)
print(config.stdout)

print("\n--- Git Status ---")
status = subprocess.run(['git', 'status'], capture_output=True, text=True)
print(status.stdout)

print("\n--- Recent Logs ---")
logs = subprocess.run(['heroku', 'logs', '-n', '50', '-a', app_name], capture_output=True, text=True)
print(logs.stdout)

## Step 11: Management Tools
Optional commands to manage your Heroku app directly from Colab.

In [ ]:
# Restart Dynos
# !heroku ps:restart -a {app_name}

In [ ]:
# View Live Logs (streams continuously, stop cell manually)
# !heroku logs --tail -a {app_name}

In [ ]:
# Delete Application (DANGER)
# !heroku apps:destroy -a {app_name} --confirm {app_name}